# Downloading a basis set from the Basis Set Exchange (BSE)

A wide range of standard basis sets used in quantum chemistry is available today through the Basis Set Exchange. 
Depending on the application, one can download basis sets of different quality levels for essentially all elements in the periodic table. In our examples, we use for testing purposes a minimal STO-3G basis set but implement the funcionality such that any available basis set can be used. The BSE provides large variety of basis set formats that can be used in common quantum chemical packages. The BSE also provides a Python API that can be used to download basis sets directly from Python code. In the following, we will download the STO-3G basis sets for C,H,O,N elements, save it on the disk and reuse for subsequent calculations. The available basis sets and their formats on the BSE website can be viewed at: https://www.basissetexchange.org/.


In [ ]:
%pip install -e /Users/rolandmitric/WORK/GITHUB/master_programming_2026

### Class representing electron shells

We start by defining a simple class to represent an electron shell (s-shell, p-shell, d-shell etc...). The class attributes should include the angular momentum quantum number `l`, the exponents of the Gaussian functions, and the coefficients of the linear combination of Gaussian functions that make up the basis function for the particular shell. Later on, this will allow us to build one and two-electron integrals arrays by looping over the shells and their attributes.

` 

In [ ]:
import numpy as np
from dataclasses import dataclass, field

# (2k-1)!! for k = 0..12  -> max angular momentum index 12
ODD_DF = np.array([
    1, 1, 3, 15, 105, 945, 10395, 135135,
    2027025, 34459425, 654729075, 13749310575, 316234143225
], dtype=np.float64)

PI = np.pi

def l_to_ijk(L):
    IJK = []
    for I in range(L, -1, -1):
        for J in range(L - I, -1, -1):
            IJK.append((I, J, L - I - J))
    return sorted(IJK, reverse=True)

def primitive_cart_norm(alpha, l, m, n):
    # normalized for (x-Ax)^l (y-Ay)^m (z-Az)^n * exp(-alpha r^2)
    alpha = np.float64(alpha)
    L = l + m + n
    num = (2.0 * alpha / PI) ** 0.75 * (4.0 * alpha) ** (0.5 * L)
    den = np.sqrt(ODD_DF[l] * ODD_DF[m] * ODD_DF[n])
    return num / den

@dataclass 
class Shell:
    l: int
    exponents: np.ndarray
    coefficients: np.ndarray
    norm_factors: np.ndarray  = field(init=False)
    center: np.ndarray = field(init=False)

    def __post_init__(self) -> None:
        object.__setattr__(self, "exponents", np.asarray(self.exponents, dtype=float))
        object.__setattr__(self, "coefficients", np.asarray(self.coefficients, dtype=float))
        self.norm_factors = self.get_norm_factors()

    def get_norm_factors(self) -> np.ndarray:
        ijk = l_to_ijk(self.l)
        norm_factors = np.empty((len(self.exponents), len(ijk)), dtype=np.float64)
        for i, (l, m, n) in enumerate(ijk):
            norm_factors[:, i] = primitive_cart_norm(self.exponents, l, m, n)
        return norm_factors


### Basis set class and handling functions

We develop a simple class to represent a basis set and a function to download the basis set from the BSE. We will use the `requests` library to make HTTP requests to the BSE API and the `json` library to parse the response. We will also use the `dataclasses` module to define a simple data structure for our basis set and the atomic shells. Later on, the code developed here will be transfered to a separate module that can be imported and used in other notebooks. In order to access the features of the Atom and Molecule classes, we install the current state of our package from the source code in editable mode. 

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Sequence 
import numpy as np
from theochem2026 import ATOMIC_NUMBERS, ELEMENT_SYMBOLS
import requests
import os
import json

@dataclass 
class BasisSet:
    name: str
    elements: dict[str, list[Shell]] = field(default_factory=dict)

    def download_from_bse(self, element_list, timeout: int = 30):
        url = f"https://www.basissetexchange.org/api/basis/{self.name}/format/json/"
        url += f"?elements={','.join(element_list)}"
        response = requests.get(url, timeout = timeout)
        response.raise_for_status()
        data_json = response.json()
        self.parse_elements(data_json)
        json.dump(data_json, open(f"{self.name}.json", "w"), indent=4)

    def parse_elements(self, data: dict) -> None:
        self.elements = {}
        basis_data = data.get("elements", {})
        for atomic_number, element_data in basis_data.items():
            symbol = ELEMENT_SYMBOLS.get(int(atomic_number))
            shells: list[Shell] = []
            for shell_data in element_data.get("electron_shells", []):
                angular_momenta = shell_data["angular_momentum"]
                contraction_coefficients = shell_data["coefficients"]
                exponents = shell_data["exponents"]
                for l, coefficients in zip(angular_momenta, contraction_coefficients):
                    shells.append(
                        Shell(
                            l=int(l),
                            exponents=exponents,
                            coefficients=np.asarray(coefficients, dtype=float),
                        )
                    )
            self.elements[symbol] = shells

    def get_basis_set(self, element_list: Sequence[str]) -> None:
        # if name.json file exists, load it, otherwise download from BSE and save to disk
        filename = f"{self.name}.json"
        if os.path.exists(filename):
            with open(filename, "r") as f:
                data_json = json.load(f)
            self.parse_elements(data_json)
        else:
            self.download_from_bse(element_list)  # download for all elements in element_list

    def __str__(self) -> str:
        output = f"Basis set: {self.name}\n"
        for symbol, shells in self.elements.items():
            output += f"Element: {symbol}\n"
            for shell in shells:
                output += f"  l={shell.l}, exponents={shell.exponents}, coefficients={shell.coefficients}\n"
        return output


In [ ]:
bs = BasisSet("sto-3g")
bs.get_basis_set(["C","H","O","N","P"])
print(bs)